# PredictShield AI - GNN / GAT Training (Attack Path Prediction) - v4

Improved from v3 (ROC-AUC 0.95 but attack precision 0.20). Four fixes:

1. **Threshold tuning** - sweep threshold on validation (v3 used fixed 0.5, which hurt precision)
2. **Edge deduplication** - aggregate duplicate (src, dst) pairs into ONE edge (v3 had 100+ duplicate edges)
3. **Richer node features** - 16 features instead of 8 (more signal per device)
4. **Balanced pos_weight** - sqrt ratio instead of full ratio (v3's 48:1 was too aggressive)

Data: `cicids2017_generated_combined.csv` (IPs needed for graph). Monday skipped.

In [1]:
import sys, subprocess
try:
    import torch, torch_geometric
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric", "--no-deps"])
    import torch, torch_geometric
print(f'PyTorch {torch.__version__}  PyG {torch_geometric.__version__}  CUDA {torch.cuda.is_available()}')


PyTorch 2.11.0+cu128  PyG 2.8.1  CUDA True


In [2]:
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score, confusion_matrix, precision_score, recall_score
import warnings; warnings.filterwarnings('ignore')
print('imports OK')


imports OK


In [3]:
WINDOW_SECONDS = 5
NODE_FEAT_DIM = 16
HIDDEN = 32
EPOCHS = 40
LR = 0.001
BATCH_SIZE = 32
RANDOM_STATE = 42

torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


Device: cuda


In [4]:
CSV_PATH = '/kaggle/input/datasets/pritammandal143/cicids2017-generated/cicids2017_generated_combined.csv'
import os
print(f'File exists: {os.path.exists(CSV_PATH)}')
if os.path.exists(CSV_PATH): print(f'Size: {os.path.getsize(CSV_PATH)/1e6:.1f} MB')


File exists: True
Size: 1530.3 MB


In [5]:
def build_graphs(csv_path, window_seconds=WINDOW_SECONDS):
    """Build graphs with 16 node features + deduplicated edges. Monday skipped."""
    graphs = []

    cols = [
        'Source IP', 'Destination IP', 'Source Port', 'Destination Port',
        'Protocol', 'Timestamp', 'Flow Duration',
        'Total Fwd Packets', 'Total Backward Packets',
        'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
        'Average Packet Size', 'Max Packet Length',
        'SYN Flag Count', 'RST Flag Count', 'FIN Flag Count', 'ACK Flag Count',
        'Label'
    ]
    df = pd.read_csv(csv_path, usecols=cols).dropna()
    df['is_attack'] = (df['Label'] != 'BENIGN').astype(int)

    df['t'] = pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
    df['t'] = df['t'].fillna(pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M', errors='coerce'))
    df = df.dropna(subset=['t'])
    df['t'] = df['t'].astype('datetime64[s]').astype('int64')
    df['window'] = df['t'] // window_seconds
    df['day'] = pd.to_datetime(df['t'], unit='s').dt.day
    df = df[df['day'] != 3]  # skip Monday (all benign)

    print(f'Flows after skip Monday: {len(df):,}')

    for _, group in df.groupby('window'):
        if len(group) < 2:
            continue

        ips = pd.unique(group[['Source IP', 'Destination IP']].values.ravel())
        ip_to_id = {ip: i for i, ip in enumerate(ips)}
        n = len(ips)

        # 16 node features per IP
        feat = np.zeros((n, NODE_FEAT_DIM), dtype=np.float32)
        for ip, i in ip_to_id.items():
            m = (group['Source IP'] == ip) | (group['Destination IP'] == ip)
            sub = group[m]
            ports = pd.unique(sub[['Source Port', 'Destination Port']].values.ravel())
            peers = pd.unique(sub[['Source IP', 'Destination IP']].values.ravel())
            total_bytes = sub['Total Length of Fwd Packets'].sum() + sub['Total Length of Bwd Packets'].sum()
            total_pkts = sub['Total Fwd Packets'].sum() + sub['Total Backward Packets'].sum()
            dur = sub['Flow Duration'].sum()
            feat[i, 0] = len(sub)
            feat[i, 1] = total_bytes
            feat[i, 2] = total_pkts
            feat[i, 3] = sub['Average Packet Size'].mean()
            feat[i, 4] = len(ports)
            feat[i, 5] = len(peers)
            feat[i, 6] = sub['is_attack'].mean()
            feat[i, 7] = sub['Protocol'].nunique()
            feat[i, 8] = sub['SYN Flag Count'].sum()
            feat[i, 9] = sub['RST Flag Count'].sum()
            feat[i, 10] = sub['FIN Flag Count'].sum()
            feat[i, 11] = sub['ACK Flag Count'].sum()
            feat[i, 12] = dur
            feat[i, 13] = sub['Flow Duration'].mean()
            feat[i, 14] = sub['Max Packet Length'].max()
            feat[i, 15] = 1 if any(p in (80, 443) for p in ports) else (0.5 if 22 in ports else 0.0)

        # DEDUPLICATE edges: aggregate all (src,dst) flows into ONE edge
        edge_agg = group.groupby(['Source IP', 'Destination IP']).agg(
            pkt_count=('Total Fwd Packets', 'sum'),
            bytes_sum=('Total Length of Fwd Packets', 'sum'),
            proto=('Protocol', 'first'),
        ).reset_index()

        src_ids = edge_agg['Source IP'].map(ip_to_id).values
        dst_ids = edge_agg['Destination IP'].map(ip_to_id).values
        edge_index = torch.tensor([src_ids, dst_ids], dtype=torch.long)
        edge_attr = torch.tensor(
            edge_agg[['pkt_count', 'bytes_sum', 'proto']].values, dtype=torch.float32
        )

        y = np.zeros(n, dtype=np.float32)
        for ip, i in ip_to_id.items():
            if group[(group['Source IP'] == ip) | (group['Destination IP'] == ip)]['is_attack'].sum() > 0:
                y[i] = 1.0

        graphs.append(Data(x=torch.tensor(feat), edge_index=edge_index, edge_attr=edge_attr, y=torch.tensor(y)))

    return graphs


graphs = build_graphs(CSV_PATH)
print(f'Built {len(graphs)} graphs')
attack_graphs = sum(1 for g in graphs if g.y.sum() > 0)
print(f'Attack graphs: {attack_graphs} / {len(graphs)}')
if graphs:
    g = graphs[0]
    print(f'Example: {g.x.shape[0]} nodes, {g.edge_index.shape[1]} edges (deduped)')


Flows after skip Monday: 2,298,395
Built 1964 graphs
Attack graphs: 520 / 1964
Example: 36 nodes, 43 edges (deduped)


In [6]:
all_x = np.concatenate([g.x.numpy() for g in graphs], axis=0)
scaler = StandardScaler().fit(all_x)
for g in graphs:
    g.x = torch.tensor(scaler.transform(g.x.numpy()), dtype=torch.float32)
print(f'Normalized {len(all_x):,} nodes')


Normalized 265,011 nodes


In [7]:
class GraphDataset(Dataset):
    def __init__(self, graphs): super().__init__(); self.graphs = graphs
    def len(self): return len(self.graphs)
    def get(self, idx): return self.graphs[idx]

n_train = int(len(graphs) * 0.7)
n_val = int(len(graphs) * 0.15)
train_graphs = graphs[:n_train]
val_graphs = graphs[n_train:n_train + n_val]
test_graphs = graphs[n_train + n_val:]

train_loader = DataLoader(GraphDataset(train_graphs), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(GraphDataset(val_graphs), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(GraphDataset(test_graphs), batch_size=BATCH_SIZE, shuffle=False)
print(f'Train {len(train_graphs)}  Val {len(val_graphs)}  Test {len(test_graphs)}')


Train 1374  Val 294  Test 296


In [8]:
class AttackPathGAT(torch.nn.Module):
    def __init__(self, in_channels=NODE_FEAT_DIM, hidden=HIDDEN, out_channels=1):
        super().__init__()
        self.g1 = GATConv(in_channels, hidden, heads=8, dropout=0.3)
        self.g2 = GATConv(hidden * 8, hidden, heads=4, dropout=0.3)
        self.g3 = GATConv(hidden * 4, out_channels, heads=1, concat=False)
    def forward(self, x, edge_index, edge_attr=None):
        x = F.elu(self.g1(x, edge_index))
        x = F.elu(self.g2(x, edge_index))
        x = self.g3(x, edge_index)
        return x.squeeze(-1)

model = AttackPathGAT().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

import math
n_pos = sum(int(g.y.sum()) for g in train_graphs)
n_neg = sum(int((g.y == 0).sum()) for g in train_graphs)
pw = math.sqrt(n_neg / max(n_pos, 1))
pos_weight = torch.tensor([pw], device=DEVICE)
print(f'pos_weight = {pw:.2f} (neg={n_neg}, pos={n_pos})')
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(model)


pos_weight = 17.61 (neg=191583, pos=618)
AttackPathGAT(
  (g1): GATConv(16, 32, heads=8)
  (g2): GATConv(256, 32, heads=4)
  (g3): GATConv(128, 1, heads=1)
)


In [9]:
def get_scores(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            out = model(batch.x, batch.edge_index, batch.edge_attr)
            preds.append(torch.sigmoid(out).cpu())
            labels.append(batch.y.cpu())
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = loss_fn(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    val_preds, val_labels = get_scores(model, val_loader)
    val_f1 = f1_score(val_labels, (val_preds > 0.5).astype(int), zero_division=0)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS}  loss={total_loss/len(train_loader):.4f}  val_f1@0.5={val_f1:.4f}')


Epoch   1/40  loss=0.1548  val_f1@0.5=0.4410
Epoch   5/40  loss=0.0443  val_f1@0.5=0.6224
Epoch  10/40  loss=0.0365  val_f1@0.5=0.5535
Epoch  15/40  loss=0.0294  val_f1@0.5=0.6094
Epoch  20/40  loss=0.0258  val_f1@0.5=0.7449
Epoch  25/40  loss=0.0259  val_f1@0.5=0.6972
Epoch  30/40  loss=0.0205  val_f1@0.5=0.6887
Epoch  35/40  loss=0.0188  val_f1@0.5=0.6944
Epoch  40/40  loss=0.0181  val_f1@0.5=0.6000


In [10]:
val_scores, val_labels = get_scores(model, val_loader)
best_t, best_f1 = 0.5, 0.0
print('Threshold sweep (val):')
for t in np.arange(0.05, 0.95, 0.01):
    f1 = f1_score(val_labels, (val_scores > t).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_t = f1, t
print(f'Best threshold: {best_t:.3f}  val F1: {best_f1:.4f}')


Threshold sweep (val):
Best threshold: 0.940  val F1: 0.7449


In [11]:
test_scores, test_labels = get_scores(model, test_loader)
test_pred = (test_scores > best_t).astype(int)

print(f'GAT Results (threshold={best_t:.3f}):')
print(f'  Accuracy : {((test_pred == test_labels).mean()):.4f}')
print(f'  Precision: {precision_score(test_labels, test_pred, zero_division=0):.4f}')
print(f'  Recall   : {recall_score(test_labels, test_pred, zero_division=0):.4f}')
print(f'  F1       : {f1_score(test_labels, test_pred, zero_division=0):.4f}')
print(f'  ROC-AUC  : {roc_auc_score(test_labels, test_scores):.4f}')
print()
print(classification_report(test_labels, test_pred, digits=3, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(test_labels, test_pred))


GAT Results (threshold=0.940):
  Accuracy : 0.9860
  Precision: 0.6759
  Recall   : 0.6023
  F1       : 0.6370
  ROC-AUC  : 0.9644

              precision    recall  f1-score   support

         0.0      0.992     0.994     0.993     37002
         1.0      0.676     0.602     0.637       772

    accuracy                          0.986     37774
   macro avg      0.834     0.798     0.815     37774
weighted avg      0.985     0.986     0.986     37774

Confusion matrix:
[[36779   223]
 [  307   465]]


In [12]:
torch.save(model.state_dict(), '/kaggle/working/gat_attack_path_state.pt')
print('Saved: /kaggle/working/gat_attack_path_state.pt')


Saved: /kaggle/working/gat_attack_path_state.pt
